In [ ]:
# from cacti.utils.config import Config
# from cacti.models.builder import build_model
# from cacti.utils.utils import save_single_image,get_device_info,load_checkpoints
# from cacti.utils.metrics import compare_psnr,compare_ssim

import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torch.cuda.amp import autocast

import numpy as np
import os
import os.path as osp
import scipy.io as scio 
import einops
import time


In [ ]:
def generate_masks_wtype(mask_path=None,mask_shape=None,mask_type='zp'):
    assert mask_path is not None or mask_shape is not None
    if mask_path is None:
        if mask_type == 'zp':
            mask = np.random.randint(0,2,size=(mask_shape[0],mask_shape[1],mask_shape[2]))
        else:
            mask0 = np.random.randint(0,2,size=(mask_shape[0],mask_shape[1],mask_shape[2]//2))
            mask1 = 1 - mask0
            mask = np.concatenate([mask0[...,np.newaxis],mask1[...,np.newaxis]],axis=-1)
            mask = mask.reshape(mask_shape[0],mask_shape[1],mask_shape[2])
    else:
        mask = scio.loadmat(mask_path)
        mask = mask['mask']
        if mask_shape is not None:
            h,w,c = mask.shape
            m_h,m_w,m_c = mask_shape[0],mask_shape[1],mask_shape[2]
            h_b = np.random.randint(0,h-m_h+1)
            w_b = np.random.randint(0,w-m_w+1)
            mask = mask[h_b:h_b+m_h,w_b:w_b+m_w,:m_c]
    mask = np.transpose(mask, [2, 0, 1])
    mask = mask.astype(np.float32)
    if mask_type == 'zp':
        mask_s = np.sum(mask, axis=0)
        mask_s[mask_s==0] = 1
    elif mask_type == 'np':
        mask_s = np.sum(mask, axis=0)
    return mask, mask_s